In [1]:
import json
import os
import sys
sys.path.insert(0, ".")  # ensure repo root is on path

# Required by nl2structnl_fretish.py to load fretish_structnl_to_ltl_dict.json
os.environ["DATA_HOME_DIR"] = os.path.abspath("metadata")

from nl2structnl_fretish import get_structNL_prompt_simple

In [2]:
ap_system_prompt = (
    "You are an expert in requirements engineering and formal specification. "
    "Given a list of natural language requirements, identify all atomic boolean propositions "
    "(system state variables) needed to express them formally as FRETish requirements. "
    "For each proposition provide a variable name and a brief description. "
    "Variable names must be valid identifiers in the FRET requirements grammar: they must "
    "start with a letter and contain only letters, digits, and underscores (no spaces, "
    "hyphens, or other special characters), and must not be one of FRET's reserved words "
    "(e.g. shall, when, if, mode, and, or, not, true, false, until, within). "
    "Use lowercase snake_case names for ordinary variables (e.g. \"sensor_is_active\"). "
    "For a variable that represents a finite-state-machine being in a particular mode, "
    "use the pattern \"state_is_<MODE_NAME>\" with the mode name in upper case "
    "(e.g. \"state_is_NOMINAL\", \"state_is_FAULT\"). "
    "IMPORTANT: requirements often give the exact variable name to use in parentheses, "
    "e.g. \"the autopilot is requesting support (request)\" or \"limits are not exceeded "
    "(not limits)\". When a requirement contains such a hint, you MUST use that hint "
    "(stripped of words like \"not\"/\"is\"/\"are\", lowercased) as the variable_name "
    "verbatim, instead of inventing a longer descriptive name. Only invent a new "
    "snake_case name for concepts that have no such hint. "
    "Generate one entry per distinct concept; do not generate duplicate or near-duplicate "
    "propositions for the same concept. "
    "Respond with a single JSON object only, with no extra text, commentary, or markdown "
    "code fences, in exactly this format:\n"
    '{"atomic_propositions": [{"variable_name": "request", '
    '"description": "True when the autopilot is requesting support"}, '
    '{"variable_name": "state_is_NOMINAL", '
    '"description": "True when the system is in the NOMINAL state"}]}'
)

# Example requirements list (normally loaded from PlausibleSpecs.xlsx)
example_nl_requirements = [
    "When the autopilot is requesting support (request), the system shall enter FAULT mode.",
    "If limits are not exceeded (not limits), the system shall remain in NOMINAL mode.",
]

ap_user_prompt = (
    "Generate atomic propositions for the following requirements:\n"
    + json.dumps(example_nl_requirements, indent=2)
)

print("=== SYSTEM PROMPT ===\n")
print(ap_system_prompt)
print("\n\n=== USER PROMPT ===\n")
print(ap_user_prompt)

=== SYSTEM PROMPT ===

You are an expert in requirements engineering and formal specification. Given a list of natural language requirements, identify all atomic boolean propositions (system state variables) needed to express them formally as FRETish requirements. For each proposition provide a variable name and a brief description. Variable names must be valid identifiers in the FRET requirements grammar: they must start with a letter and contain only letters, digits, and underscores (no spaces, hyphens, or other special characters), and must not be one of FRET's reserved words (e.g. shall, when, if, mode, and, or, not, true, false, until, within). Use lowercase snake_case names for ordinary variables (e.g. "sensor_is_active"). For a variable that represents a finite-state-machine being in a particular mode, use the pattern "state_is_<MODE_NAME>" with the mode name in upper case (e.g. "state_is_NOMINAL", "state_is_FAULT"). IMPORTANT: requirements often give the exact variable name to 

In [3]:
example_input_nl = "When the autopilot is requesting support, the system shall immediately enter FAULT mode."

example_ap_dict = {
    "request": "True when the autopilot is requesting support",
    "state_is_FAULT": "True when the system is in the FAULT state",
}

formalization_system_prompt, formalization_user_prompt = get_structNL_prompt_simple(
    example_input_nl,
    example_ap_dict,
    dcmp=None,
    k=10,
)

print("=== SYSTEM PROMPT ===\n")
print(formalization_system_prompt)
print("\n\n=== USER PROMPT ===\n")
print(formalization_user_prompt)

=== SYSTEM PROMPT ===

You are an expert in Linear Temporal Logic and requirements engineering. Your job is to translate natural language requirements to structured natural language that capture the intents of the requirements.


To produce the structured natural language property, you compose it from a set of templates.
If the chosen option contains boolean expression placeholders (i.e., bool_exp1, bool_exp2, bool_exp3, bool_exp4), you need to produce boolean expressions that will replace the placeholders.
Boolean expressions can only contain boolean operators (e.g., !, &, |, ->, <->) and can only atomic propositions (NO NUMERICAL COMPARISON OPERATORS ALLOWED)

The following lists the ONLY valid options for decision1, decision2, and decision3. You MUST copy one of these strings exactly — do not paraphrase or invent new values:

decision1_options = [
    "while bool_exp1, _ABSTRACT_VAR1_",
    "before bool_exp1, _ABSTRACT_VAR1_",
    "after bool_exp1, _ABSTRACT_VAR1_",
    "whenever bo